# Ch.2 — Boosting: AdaBoost & Gradient Boosting

## EnsembleAI Challenge — Ch.2

Ch.1 showed **bagging** reduces variance. This chapter: **boosting** reduces **bias**.

- **AdaBoost**: Reweight misclassified samples → next learner focuses on hard cases
- **Gradient Boosting**: Each tree fits the *residuals* of the current ensemble

## Setup & Data

In [ ]:
# ── Setup & Data ─────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import (AdaBoostClassifier, GradientBoostingRegressor,
                               GradientBoostingClassifier, RandomForestRegressor)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import (mean_squared_error, r2_score,
                              f1_score, accuracy_score)

IMG = "img/"
import os; os.makedirs(IMG, exist_ok=True)

np.random.seed(42)

data          = fetch_california_housing()
X, y_reg      = data.data, data.target
feature_names = list(data.feature_names)
y_cls         = (y_reg > np.median(y_reg)).astype(int)

(X_train, X_test,
 y_tr, y_te,
 y_tr_cls, y_te_cls) = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42)

print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")

## AdaBoost — Sequential Sample Reweighting

In [ ]:
# ── AdaBoost Classification ──────────────────────────────────────────────────
ada_stumps = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # stumps
    n_estimators=200, learning_rate=0.5, random_state=42)
ada_stumps.fit(X_train, y_tr_cls)

ada_trees = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=3),  # deeper trees
    n_estimators=200, learning_rate=0.5, random_state=42)
ada_trees.fit(X_train, y_tr_cls)

dt_base = DecisionTreeClassifier(max_depth=1, random_state=42)
dt_base.fit(X_train, y_tr_cls)

print(f"Single stump F1:      {f1_score(y_te_cls, dt_base.predict(X_test)):.4f}")
print(f"AdaBoost (stumps) F1: {f1_score(y_te_cls, ada_stumps.predict(X_test)):.4f}")
print(f"AdaBoost (depth=3) F1:{f1_score(y_te_cls, ada_trees.predict(X_test)):.4f}")

## AdaBoost Staged Performance

In [ ]:
# ── AdaBoost: accuracy vs boosting rounds ────────────────────────────────────
staged_scores = []
for y_pred in ada_stumps.staged_predict(X_test):
    staged_scores.append(f1_score(y_te_cls, y_pred))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, len(staged_scores)+1), staged_scores, color='steelblue', linewidth=2)
ax.axhline(f1_score(y_te_cls, dt_base.predict(X_test)), color='coral',
           linestyle='--', label='Single stump baseline')
ax.set_xlabel('Boosting round'); ax.set_ylabel('Test F1')
ax.set_title('AdaBoost — F1 vs Boosting Rounds', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(f'{IMG}ch02_adaboost_staged.png', dpi=150, bbox_inches='tight')
plt.show()
print("Weak stumps → strong classifier via boosting.")

## Gradient Boosting — Fitting Residuals

Each tree fits: $h_m(\mathbf{x}) \approx y - F_{m-1}(\mathbf{x})$ (the residual).

Update: $F_m = F_{m-1} + \eta \cdot h_m$

In [ ]:
# ── Gradient Boosting Regression ─────────────────────────────────────────────
gb = GradientBoostingRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=4,
    subsample=0.8, random_state=42,
    validation_fraction=0.15, n_iter_no_change=30, tol=1e-4)
gb.fit(X_train, y_tr)

rmse_gb = np.sqrt(mean_squared_error(y_te, gb.predict(X_test)))
print(f"Gradient Boosting RMSE: {rmse_gb:.4f}")
print(f"Stopped at iteration: {gb.n_estimators_}")

## Visualize Residuals Shrinking

In [ ]:
# ── Residual Shrinkage Across Boosting Rounds ────────────────────────────────
staged_rmse = []
for y_pred in gb.staged_predict(X_test):
    staged_rmse.append(np.sqrt(mean_squared_error(y_te, y_pred)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, len(staged_rmse)+1), staged_rmse, color='steelblue', linewidth=1.5)
ax.axvline(gb.n_estimators_, color='coral', linestyle='--',
           label=f'Early stop at {gb.n_estimators_}')
ax.set_xlabel('Boosting round'); ax.set_ylabel('Test RMSE')
ax.set_title('Gradient Boosting — RMSE Drops Each Round', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(f'{IMG}ch02_gb_residual_shrinkage.png', dpi=150, bbox_inches='tight')
plt.show()

## Learning Rate Trade-off

In [ ]:
# ── Learning Rate Effect ─────────────────────────────────────────────────────
lr_values = [0.3, 0.1, 0.05, 0.01]
fig, ax = plt.subplots(figsize=(9, 4))

for lr in lr_values:
    gb_lr = GradientBoostingRegressor(
        n_estimators=500, learning_rate=lr, max_depth=4,
        random_state=42, validation_fraction=0.15, n_iter_no_change=50)
    gb_lr.fit(X_train, y_tr)
    staged = [np.sqrt(mean_squared_error(y_te, p)) for p in gb_lr.staged_predict(X_test)]
    ax.plot(range(1, len(staged)+1), staged, linewidth=1.5,
            label=f'lr={lr} (stop={gb_lr.n_estimators_}, RMSE={staged[-1]:.4f})')

ax.set_xlabel('Boosting round'); ax.set_ylabel('Test RMSE')
ax.set_title('Learning Rate vs Convergence — Lower lr = slower but better', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(f'{IMG}ch02_learning_rate_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

## Bagging vs Boosting Benchmark

In [ ]:
# ── RF vs GB Comparison ──────────────────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=200, max_features='sqrt',
                           random_state=42, n_jobs=-1)
rf.fit(X_train, y_tr)
rmse_rf = np.sqrt(mean_squared_error(y_te, rf.predict(X_test)))

dt = DecisionTreeRegressor(max_depth=None, random_state=42)
dt.fit(X_train, y_tr)
rmse_dt = np.sqrt(mean_squared_error(y_te, dt.predict(X_test)))

print(f"{'Model':<30} {'RMSE':>8} {'R²':>8}")
print("-" * 50)
for name, pred_fn in [('Decision Tree', dt), ('Random Forest (200)', rf), ('Gradient Boosting', gb)]:
    preds = pred_fn.predict(X_test)
    print(f"{name:<30} {np.sqrt(mean_squared_error(y_te, preds)):>8.4f} {r2_score(y_te, preds):>8.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
models = ['Decision Tree', 'Random Forest', 'Gradient Boost']
rmses = [rmse_dt, rmse_rf, rmse_gb]
colors = ['#aec6e8', '#6baed6', '#2171b5']
ax.bar(models, rmses, color=colors, edgecolor='white')
ax.set_ylabel('Test RMSE'); ax.set_title('DT vs RF (bagging) vs GB (boosting)', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(f'{IMG}ch02_bagging_vs_boosting.png', dpi=150, bbox_inches='tight')
plt.show()

## What Can Go Wrong: Boosting on Noisy Labels

In [ ]:
# ── Noise Overfitting Demo ───────────────────────────────────────────────────
rng = np.random.default_rng(42)
y_tr_noisy = y_tr.copy()
noise_idx  = rng.choice(len(y_tr_noisy), size=int(0.15 * len(y_tr_noisy)), replace=False)
y_tr_noisy[noise_idx] += rng.normal(0, 2.0, len(noise_idx))

gb_noisy = GradientBoostingRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=4, random_state=42)
gb_noisy.fit(X_train, y_tr_noisy)

gb_noisy_es = GradientBoostingRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=4, random_state=42,
    validation_fraction=0.15, n_iter_no_change=30)
gb_noisy_es.fit(X_train, y_tr_noisy)

rmse_no_es = np.sqrt(mean_squared_error(y_te, gb_noisy.predict(X_test)))
rmse_es    = np.sqrt(mean_squared_error(y_te, gb_noisy_es.predict(X_test)))

print(f"15% noisy labels:")
print(f"  Without early stopping (500 rounds): RMSE = {rmse_no_es:.4f}")
print(f"  With early stopping (stopped at {gb_noisy_es.n_estimators_}): RMSE = {rmse_es:.4f}")
print(f"  Clean data GB: RMSE = {rmse_gb:.4f}")

## Progress Check

| # | Constraint | Status | Evidence |
|---|-----------|--------|----------|
| 1 | IMPROVEMENT >5% | ✅ | GB beats RF on RMSE |
| 2 | DIVERSITY | ✅ | Sequential trees correct different errors |
| 3 | EFFICIENCY | ⏳ | Sequential training is slower |
| 4 | INTERPRETABILITY | ⚡ | Feature importance available |
| 5 | ROBUSTNESS | ⚠️ | Needs early stopping on noisy data |

**Next**: Ch.3 — XGBoost, LightGBM, CatBoost (industrial-strength boosting).

## Exercises

1. **AdaBoost depth sweep**: Train AdaBoost with base tree `max_depth` in `[1, 2, 3, 5, 10]`. Plot test F1. What depth works best for boosting?
2. **Subsample effect**: Train GradientBoostingRegressor with `subsample` in `[0.5, 0.7, 0.8, 0.9, 1.0]`. Plot test RMSE. Does stochastic gradient boosting help?
3. **Bias-variance decomposition**: For the same dataset, compute RMSE for RF (bagging) vs GB (boosting) across 10 random seeds. Which has lower variance? Lower bias?

In [ ]:
# ── Exercise 1: AdaBoost depth sweep ─────────────────────────────────────────
# TODO: for depth in [1, 2, 3, 5, 10]:
#     ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=depth), ...)
#     ada.fit(X_train, y_tr_cls)
#     record F1
pass

In [ ]:
# ── Exercise 2: Subsample effect ─────────────────────────────────────────────
# TODO: for ss in [0.5, 0.7, 0.8, 0.9, 1.0]:
#     gb_ss = GradientBoostingRegressor(subsample=ss, ...)
#     gb_ss.fit(X_train, y_tr)
#     record RMSE
pass

In [ ]:
# ── Exercise 3: Bias-variance across seeds ───────────────────────────────────
# TODO: for seed in range(10):
#     train RF and GB with random_state=seed
#     record RMSE
# Compare std(RMSE) for RF vs GB
pass